In [3]:
pip install pyreadstat

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------  2.4/2.4 MB 12.2 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 11.4 MB/s eta 0:00:00

  Attempting uninstall: narwhals

    Found existing installation: narwhals 1.47.1

    Uninstalling narwhals-1.47.1:

      Successfully uninstalled narwhals-1.47.1

   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   -------------------- ------------------- 1/2 [pyreadstat]
   ---------------------------------------- 2/2 [pyreadstat]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# 5개 영화들만 추출(5개만 했는데 결과 잘 안나옴)

In [13]:
import pandas as pd
import pyreadstat

# 데이터 불러오기
df = pd.read_csv("movie_long_panel_2023_2024.csv", parse_dates=["DATE","일자","개봉일"])

# 댓글 수 상위 5개 영화 추출
top_movies = (
    df.groupby("영화명")["감상평"].apply(lambda x: x.notna().sum())
    .sort_values(ascending=False)
    .head(5)
    .index.tolist()
)
print("선택된 TOP 5 영화:", top_movies)

df_top = df[df["영화명"].isin(top_movies)].copy()

# OTT 공개일 = post_netflix == 1 첫 날짜
ott_release = (
    df_top[df_top["post_netflix"] == 1]
    .groupby("영화명")["DATE"].min()
    .reset_index()
    .rename(columns={"DATE":"OTT공개일"})
)
df_top = df_top.merge(ott_release, on="영화명", how="left")

# treatment 변수 생성
df_top["treatment"] = (
    (df_top["OTT공개일"].notna()) & (df_top["DATE"] >= df_top["OTT공개일"])
).astype(int)



선택된 TOP 5 영화: ['콘크리트 유토피아', '노량: 죽음의 바다', '그대들은 어떻게 살 것인가', '더 문', '잠']


C:\Users\82104\AppData\Local\Temp\ipykernel_25300\3479487922.py:5: DtypeWarning: Columns (12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("movie_long_panel_2023_2024.csv", parse_dates=["DATE","일자","개봉일"])


In [17]:
# numeric time (Stata 기준 1960-01-01)
df_top["date_num"] = (df_top["DATE"] - pd.Timestamp("1960-01-01")).dt.days

# outcome 후보: 감정 확률 변수들
prob_cols = [c for c in df_top.columns if c.endswith("_prob")]

# 최종 보존할 변수
keep_cols = ["영화명","DATE","date_num","개봉일","post_netflix","OTT공개일","treatment"] + prob_cols
df_stata = df_top[keep_cols].copy()


In [19]:
import re

def clean_name(name):
    # 한글/영문/숫자/밑줄만 남기고 나머지는 밑줄로 치환
    name = re.sub(r"[^0-9a-zA-Z가-힣_]", "_", name)
    # 길이 32자 제한
    return name[:32]

df_stata = df_stata.rename(columns={c: clean_name(c) for c in df_stata.columns})

In [21]:
# .dta 저장
pyreadstat.write_dta(df_stata, "movie_long_panel_top5_emotions.dta")
print("✅ 저장 완료 → movie_long_panel_top5_emotions.dta")

✅ 저장 완료 → movie_long_panel_top5_emotions.dta


# 전체 영화 전처리

In [14]:
import pandas as pd

# ----- 1. 데이터 불러오기 -----
print("데이터를 불러오는 중입니다...")
input_filename = 'movie_long_with_emotions_full.csv'
# 인코딩 에러 방지를 위해 encoding='cp949' 추가
df = pd.read_csv(input_filename, encoding='cp949')
print("데이터 불러오기 완료.")


# ----- 2. ✨새로운 로직: post_netflix 기준으로 OTT 공개일 찾기✨ -----
print("post_netflix가 1이 되는 첫 날짜를 OTT 공개일로 정의합니다...")
# 날짜 형식으로 변환
df['DATE'] = pd.to_datetime(df['DATE'], errors='coerce')
df.dropna(subset=['DATE'], inplace=True)

# post_netflix가 1인 데이터만 필터링
df_ott_first_day = df[df['post_netflix'] == 1].copy()

# 각 영화별로 post_netflix가 1이 된 가장 빠른 날짜를 찾습니다.
ott_release_dates = df_ott_first_day.groupby('영화명')['DATE'].min().reset_index()
# 찾은 날짜 정보를 'OTT공개일'이라는 새 컬럼 이름으로 지정합니다.
ott_release_dates.rename(columns={'DATE': 'OTT공개일'}, inplace=True)

# 원본 데이터에 새로 정의한 'OTT공개일' 정보를 병합합니다.
df.drop(columns=['OTT공개일'], inplace=True, errors='ignore') # 기존에 있었다면 삭제
df = pd.merge(df, ott_release_dates, on='영화명', how='left')

# OTT 공개일이 없는 영화(post_netflix가 1이 된 적 없는 영화)는 분석에서 제외합니다.
df.dropna(subset=['OTT공개일'], inplace=True)
print("OTT 공개일 정의 및 병합 완료.")

데이터를 불러오는 중입니다...
데이터 불러오기 완료.
post_netflix가 1이 되는 첫 날짜를 OTT 공개일로 정의합니다...
OTT 공개일 정의 및 병합 완료.


In [16]:
# ----- 3. 2023-2024년 데이터 필터링 -----
print("2023년 1월 1일부터 2024년 12월 31일까지 데이터 필터링 중...")
df_filtered = df[(df['DATE'] >= '2023-01-01') & (df['DATE'] <= '2024-12-31')].copy()

# 감정 확률 변수 목록 정의
emotion_prob_columns = [col for col in df.columns if col.endswith('_prob')]


# ----- 4. 일별 중복 데이터 평균내기 -----
print("일별 중복 데이터를 평균으로 집계 중입니다...")
agg_dict = {col: 'mean' for col in emotion_prob_columns}
for col in ['OTT공개일', '대표국적', '장르']:
    if col in df_filtered.columns:
        agg_dict[col] = 'first'
df_daily_agg = df_filtered.groupby(['영화명', 'DATE']).agg(agg_dict).reset_index()
print("일별 집계 완료.")

2023년 1월 1일부터 2024년 12월 31일까지 데이터 필터링 중...
일별 중복 데이터를 평균으로 집계 중입니다...
일별 집계 완료.


In [18]:
# ----- 5. 균형 잡힌 패널(Balanced Panel) 생성 -----
print("균형 잡힌 패널을 구성 중입니다...")
date_range = pd.date_range(start='2023-01-01', end='2024-12-31')
movie_list = df_daily_agg['영화명'].unique()
panel_skeleton = pd.MultiIndex.from_product([movie_list, date_range], names=['영화명', 'DATE'])
panel_df = pd.DataFrame(index=panel_skeleton).reset_index()

# 뼈대와 실제 데이터를 병합
df_panel = pd.merge(panel_df, df_daily_agg, on=['영화명', 'DATE'], how='left')

균형 잡힌 패널을 구성 중입니다...


In [20]:
# 메타데이터 채우기
df_panel.sort_values(by=['영화명', 'DATE'], inplace=True)
cols_to_fill = ['OTT공개일', '대표국적', '장르']
for col in cols_to_fill:
    if col in df_panel.columns:
        df_panel[col] = df_panel.groupby('영화명')[col].ffill().bfill()

df_panel.rename(columns={'DATE': 'panel_date'}, inplace=True)
print("패널 구성 완료.")

패널 구성 완료.


In [22]:

# ----- 6. Stata 파일로 저장 -----
output_filename_dta = 'movie_panel_2023_2024.dta'
df_panel.to_stata(output_filename_dta, write_index=False, version=118)

print(f"\n✨ 모든 작업 완료! ✨")
print(f"최종 결과가 Stata 파일 '{output_filename_dta}'으로 저장되었습니다.")

C:\Users\82104\AppData\Local\Temp\ipykernel_19000\3876827589.py:3: InvalidColumnName: 
Not all pandas column names were valid Stata variable names.
The following replacements have been made:

    불평/불만_prob   ->   불평_불만_prob
    환영/호의_prob   ->   환영_호의_prob
    감동/감탄_prob   ->   감동_감탄_prob
    화남/분노_prob   ->   화남_분노_prob
    우쭐댐/무시함_prob   ->   우쭐댐_무시함_prob
    안타까움/실망_prob   ->   안타까움_실망_prob
    의심/불신_prob   ->   의심_불신_prob
    편안/쾌적_prob   ->   편안_쾌적_prob
    신기함/관심_prob   ->   신기함_관심_prob
    공포/무서움_prob   ->   공포_무서움_prob
    역겨움/징그러움_prob   ->   역겨움_징그러움_prob
    패배/자기혐오_prob   ->   패배_자기혐오_prob
    힘듦/지침_prob   ->   힘듦_지침_prob
    즐거움/신남_prob   ->   즐거움_신남_prob
    증오/혐오_prob   ->   증오_혐오_prob
    흐뭇함(귀여움/예쁨)_prob   ->   흐뭇함_귀여움_예쁨__prob
    당황/난처_prob   ->   당황_난처_prob
    부담/안_내킴_prob   ->   부담_안_내킴_prob
    불쌍함/연민_prob   ->   불쌍함_연민_prob
    불안/걱정_prob   ->   불안_걱정_prob
    안심/신뢰_prob   ->   안심_신뢰_prob

If this is not what you expect, please make sure you have Stata-complian


✨ 모든 작업 완료! ✨
최종 결과가 Stata 파일 'movie_panel_2023_2024.dta'으로 저장되었습니다.


In [29]:
import pandas as pd

# ----- 1. 데이터 불러오기 -----
print("일별 패널 데이터를 불러오는 중입니다...")
# Stata 패널 데이터 파일 이름을 정확하게 입력해주세요.
input_filename = 'movie_panel_2023_2024.dta'
df = pd.read_stata(input_filename)
print("데이터 불러오기 완료.")


# ----- 2. 주(Week) 기준 변수 생성 -----
print("주(Week) 단위 변수를 생성 중입니다...")
# 날짜 컬럼들을 datetime 형식으로 변환 (오류 무시)
df['panel_date'] = pd.to_datetime(df['panel_date'], errors='coerce')
df['OTT공개일'] = pd.to_datetime(df['OTT공개일'], errors='coerce')

# 각 날짜가 속한 주의 시작일(월요일)을 계산하여 'week_start' 컬럼 생성
df['week_start'] = df['panel_date'].dt.to_period('W').apply(lambda r: r.start_time)

# 전체 데이터의 첫 주를 기준으로 주차 인덱스('week_index') 생성 (0부터 시작)
first_week = df['week_start'].min()
df['week_index'] = (df['week_start'] - first_week).dt.days // 7

# 각 영화의 OTT 공개일이 속한 주의 주차 인덱스를 'gvar_week'로 생성
df['gvar_week_start'] = df['OTT공개일'].dt.to_period('W').apply(lambda r: r.start_time)
df['gvar_week'] = (df['gvar_week_start'] - first_week).dt.days // 7
# gvar_week를 정수형으로 변환 (결측치가 있을 수 있으므로 Int64 사용)
df['gvar_week'] = df['gvar_week'].astype('Int64')
print("주(Week) 변수 생성 완료.")


# ----- 3. ✨주 단위로 데이터 집계✨ -----
print("주 단위로 데이터를 집계 중입니다...")
# 감정 확률 변수 목록을 자동으로 찾기
emotion_prob_columns = [col for col in df.columns if col.endswith('_prob')]

# 집계 규칙 정의:
# - 감정 확률 변수들은 주 평균(mean)으로 계산
# - 나머지 메타데이터는 해당 주의 첫번째(first) 값 사용
agg_dict = {col: 'mean' for col in emotion_prob_columns}
for col in ['gvar_week', '대표국적', '장르']:
    if col in df.columns:
        agg_dict[col] = 'first'

# 영화명과 주차를 기준으로 그룹화하여 집계 실행
df_weekly = df.groupby(['영화명', 'week_index', 'week_start']).agg(agg_dict).reset_index()
print("주 단위 집계 완료.")


# ----- 4. 최종 파일 저장 -----
output_filename = 'movie_weekly_panel_2023_2024.dta'
df_weekly.to_stata(output_filename, write_index=False, version=118)

print(f"\n✨ 모든 작업 완료! ✨")
print(f"최종 주별 집계 결과가 Stata 파일 '{output_filename}'으로 저장되었습니다.")

일별 패널 데이터를 불러오는 중입니다...
데이터 불러오기 완료.
주(Week) 단위 변수를 생성 중입니다...
주(Week) 변수 생성 완료.
주 단위로 데이터를 집계 중입니다...
주 단위 집계 완료.

✨ 모든 작업 완료! ✨
최종 주별 집계 결과가 Stata 파일 'movie_weekly_panel_2023_2024.dta'으로 저장되었습니다.


In [31]:
# 1. 데이터 불러오기
# 파일 이름은 실제 파일명과 동일해야 합니다.
filename = 'movie_weekly_panel_2023_2024.dta'
df = pd.read_stata(filename)

# 2. 모든 영화가 OTT에 공개되었는지 확인
# 각 영화별로 고유한 gvar_week 값을 추출합니다.
gvar_per_movie = df.groupby('영화명')['gvar_week'].first()
# gvar_week 값 중에 비어있는(null) 값이 하나라도 있는지 확인합니다.
# .isnull().any()가 False이면 모든 영화에 공개일이 있다는 의미입니다.
all_movies_have_release_date = not gvar_per_movie.isnull().any()

print(f"모든 영화가 OTT에 공개되었나요? -> {all_movies_have_release_date}")


# 3. 주차별 공개 분포 확인
# gvar_week 값들의 빈도를 계산하고 주차 순서대로 정렬합니다.
week_distribution = gvar_per_movie.value_counts().sort_index()

print("\n--- OTT 공개 주차별 영화 편수 ---")
print(week_distribution)


모든 영화가 OTT에 공개되었나요? -> True

--- OTT 공개 주차별 영화 편수 ---
gvar_week
-140    1
-115    1
-108    2
-105    2
-86     1
       ..
 113    1
 114    3
 120    3
 121    2
 122    1
Name: count, Length: 97, dtype: int64


# 월별 전처리

In [2]:
import pandas as pd

# ----- 1. 데이터 불러오기 -----
print("원본 데이터를 불러오는 중입니다...")
input_filename = 'movie_long_with_emotions_full.csv'
# 인코딩 에러 방지를 위해 encoding='cp949' 추가
df = pd.read_csv(input_filename, encoding='cp949')
print("데이터 불러오기 완료.")


# ----- 2. OTT 공개일 정의 및 기본 전처리 -----
print("OTT 공개일 정의 및 기본 전처리 중...")
df['DATE'] = pd.to_datetime(df['DATE'], errors='coerce')
df.dropna(subset=['DATE'], inplace=True)


원본 데이터를 불러오는 중입니다...
데이터 불러오기 완료.
OTT 공개일 정의 및 기본 전처리 중...


In [4]:
# post_netflix가 1이 되는 첫 날짜를 OTT 공개일로 정의
df_ott_first_day = df[df['post_netflix'] == 1].copy()
ott_release_dates = df_ott_first_day.groupby('영화명')['DATE'].min().reset_index()
ott_release_dates.rename(columns={'DATE': 'OTT공개일'}, inplace=True)

# 원본 데이터에 새로 정의한 'OTT공개일' 정보를 병합
if 'OTT공개일' in df.columns:
    df.drop(columns=['OTT공개일'], inplace=True)
df = pd.merge(df, ott_release_dates, on='영화명', how='left')

# OTT 공개일이 없는 영화는 분석에서 제외
df.dropna(subset=['OTT공개일'], inplace=True)
print("OTT 공개일 정의 완료.")


# ----- 3. 2023-2024년 데이터 필터링 -----
print("2023-2024년 데이터 필터링 중...")
df_filtered = df[(df['DATE'] >= '2023-01-01') & (df['DATE'] <= '2024-12-31')].copy()

# 감정 확률 변수 목록 정의
emotion_prob_columns = [col for col in df.columns if col.endswith('_prob')]

OTT 공개일 정의 완료.
2023-2024년 데이터 필터링 중...


In [6]:
# ----- 4. 월(Month) 단위로 데이터 집계 -----
print("데이터를 월 단위로 집계 중입니다...")
# 각 날짜가 속한 월의 시작일(예: 2023-01-01)을 계산
df_filtered['month_start'] = df_filtered['DATE'].dt.to_period('M').apply(lambda p: p.start_time)

# 집계 규칙 정의
agg_dict = {col: 'mean' for col in emotion_prob_columns} # 감정변수는 월 평균
for col in ['OTT공개일', '대표국적', '장르']:
    if col in df_filtered.columns:
        agg_dict[col] = 'first' # 메타데이터는 첫번째 값

# 영화명과 월(month_start)을 기준으로 그룹화하여 집계
df_monthly_agg = df_filtered.groupby(['영화명', 'month_start']).agg(agg_dict).reset_index()
print("월 단위 집계 완료.")


데이터를 월 단위로 집계 중입니다...
월 단위 집계 완료.


In [8]:
# ----- 5. 균형 잡힌 월별 패널(Balanced Panel) 생성 -----
print("균형 잡힌 월별 패널을 구성 중입니다...")
# 패널의 전체 기간 정의 (월 시작일 기준)
date_range = pd.date_range(start='2023-01-01', end='2024-12-31', freq='MS')
movie_list = df_monthly_agg['영화명'].unique()

# 패널의 '뼈대'를 생성
panel_skeleton = pd.MultiIndex.from_product([movie_list, date_range], names=['영화명', 'month_start'])
panel_df = pd.DataFrame(index=panel_skeleton).reset_index()

# 뼈대와 실제 월별 데이터를 병합
df_panel = pd.merge(panel_df, df_monthly_agg, on=['영화명', 'month_start'], how='left')

# 메타데이터 채우기 (리뷰가 없는 월에도 영화 정보가 있도록)
df_panel.sort_values(by=['영화명', 'month_start'], inplace=True)
cols_to_fill = ['OTT공개일', '대표국적', '장르']
for col in cols_to_fill:
    if col in df_panel.columns:
        df_panel[col] = df_panel.groupby('영화명')[col].ffill().bfill()
print("패널 구성 완료.")


균형 잡힌 월별 패널을 구성 중입니다...
패널 구성 완료.


In [10]:
# ----- 6. CSDID용 변수 생성 및 저장 -----
print("CSDID용 변수 생성 및 파일 저장 중...")
# time 변수: month_index (0~23)
first_month = df_panel['month_start'].min()
df_panel['month_index'] = ((df_panel['month_start'].dt.year - first_month.year) * 12 +
                           (df_panel['month_start'].dt.month - first_month.month))

# gvar 변수: gvar_month (OTT 공개 월의 month_index)
df_panel['gvar_month_start'] = df_panel['OTT공개일'].dt.to_period('M').apply(lambda p: p.start_time)
df_panel['gvar_month'] = ((df_panel['gvar_month_start'].dt.year - first_month.year) * 12 +
                          (df_panel['gvar_month_start'].dt.month - first_month.month))

# Stata 파일로 저장
output_filename_dta = 'movie_monthly_panel_2023_2024.dta'
df_panel.to_stata(output_filename_dta, write_index=False, version=118)

print(f"\n✨ 모든 작업 완료! ✨")
print(f"최종 결과가 Stata 파일 '{output_filename_dta}'으로 저장되었습니다.")

CSDID용 변수 생성 및 파일 저장 중...

✨ 모든 작업 완료! ✨
최종 결과가 Stata 파일 'movie_monthly_panel_2023_2024.dta'으로 저장되었습니다.


C:\Users\82104\AppData\Local\Temp\ipykernel_8128\3669108186.py:15: InvalidColumnName: 
Not all pandas column names were valid Stata variable names.
The following replacements have been made:

    불평/불만_prob   ->   불평_불만_prob
    환영/호의_prob   ->   환영_호의_prob
    감동/감탄_prob   ->   감동_감탄_prob
    화남/분노_prob   ->   화남_분노_prob
    우쭐댐/무시함_prob   ->   우쭐댐_무시함_prob
    안타까움/실망_prob   ->   안타까움_실망_prob
    의심/불신_prob   ->   의심_불신_prob
    편안/쾌적_prob   ->   편안_쾌적_prob
    신기함/관심_prob   ->   신기함_관심_prob
    공포/무서움_prob   ->   공포_무서움_prob
    역겨움/징그러움_prob   ->   역겨움_징그러움_prob
    패배/자기혐오_prob   ->   패배_자기혐오_prob
    힘듦/지침_prob   ->   힘듦_지침_prob
    즐거움/신남_prob   ->   즐거움_신남_prob
    증오/혐오_prob   ->   증오_혐오_prob
    흐뭇함(귀여움/예쁨)_prob   ->   흐뭇함_귀여움_예쁨__prob
    당황/난처_prob   ->   당황_난처_prob
    부담/안_내킴_prob   ->   부담_안_내킴_prob
    불쌍함/연민_prob   ->   불쌍함_연민_prob
    불안/걱정_prob   ->   불안_걱정_prob
    안심/신뢰_prob   ->   안심_신뢰_prob

If this is not what you expect, please make sure you have Stata-complian